<a href="https://colab.research.google.com/github/1Jaffry1/student-workshop/blob/master/01_YOLO_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modern Object Detection I: YOLO

One-stage / dense object detection.


## 0. Workshop introduction

YOLO (You Only Look Once) is a **one-stage** detector: a single CNN looks at the whole image and predicts boxes everywhere at once.

That is different from two-stage methods (next notebook), which first guess *where* objects might be and only then classify those regions.

This notebook uses **YOLOv8n** (Ultralytics). We choose v8 rather than newer NMS-free variants so you can still see classical dense prediction + NMS.


## 1. Learning objectives

By the end of this notebook you should be able to:

- Explain what “one-stage” and “dense prediction” mean.
- Name the role of the **backbone**, **neck**, and **detection head**.
- Describe why **NMS** (non-maximum suppression) is needed after dense prediction.
- Assemble the YOLO inference pipeline from provided functions.
- *(Optional)* Run the same pipeline on **video** frames — detection repeated over time.


## How this workshop is structured

You will **not** implement neural-network layers from scratch.

The instructor cells already contain working functions for each important stage of the algorithm. Your job is to:

1. Read what each stage does and why it exists.
2. Assemble those stages in the correct order (a short coding task).
3. Change one or two parameters and watch the output change.

The demo cell is only a one-liner (`run_full_pipeline`) so you can see a result after Run all. **Do not copy that function for the assembly exercise** — wire the named stages listed in the student task.

Hands-on coding is intentionally light (~20–25% of the session). Most of the time is for understanding the pipeline.


## 2. Environment setup

Use a GPU runtime. Do not install a second copy of PyTorch — Colab already has one.


In [ ]:
!pip install -q ultralytics==8.3.155 matplotlib opencv-python-headless pillow


In [ ]:
import platform
import sys

print("Python version:", sys.version.split()[0])
print("Platform:", platform.platform())

import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(props.total_memory / 1024 ** 3, 2), "GB")
else:
    print("GPU: None")
    print("GPU memory: n/a")
    print("\nEnable a GPU: Runtime → Change runtime type → T4 GPU, then Restart session.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


## Troubleshooting

| Problem | Fix |
|---|---|
| CUDA is unavailable | `Runtime → Change runtime type → T4 GPU`, then restart and Run all |
| Package import fails | `Runtime → Restart session`, then Run all |
| Checkpoint download fails | Re-run the setup / model-load cell |
| Out of memory | Use the smaller default model, or a smaller image |
| A student cell has `???` | That is expected. Fill it in, or set `RUN_STUDENT_ASSEMBLY = False` to skip it |

Do not spend workshop time debugging package conflicts. Restart and Run all first.


## 3. Imports


In [ ]:
# ==========================================
# INSTRUCTOR PROVIDED — DO NOT MODIFY
# ==========================================

import urllib.error
import urllib.request
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import patches
from PIL import Image

SAMPLE_IMAGES = {
    "bus": "https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/assets/bus.jpg",
    "zidane": "https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/assets/zidane.jpg",
    "cats": "http://images.cocodataset.org/val2017/000000039769.jpg",
    "living_room": "http://images.cocodataset.org/val2017/000000000139.jpg",
    "street": "http://images.cocodataset.org/val2017/000000037777.jpg",
}
SAMPLE_VIDEOS = {
    "demo": "https://github.com/ultralytics/assets/releases/download/v0.0.0/solutions_ci_demo.mp4",
}


def download_file(url, path="sample.bin"):
    path = Path(path)
    if path.exists():
        return path
    req = urllib.request.Request(url, headers={"User-Agent": "object-detection-workshop/1.0"})
    try:
        with urllib.request.urlopen(req) as resp:
            path.write_bytes(resp.read())
    except urllib.error.HTTPError as err:
        loc = err.headers.get("Location")
        if err.code in (301, 302, 303, 307, 308) and loc:
            return download_file(loc, path)
        raise
    return path


def download_image(url, path="sample.jpg"):
    return download_file(url, path)


def download_video(url, path="sample.mp4"):
    return download_file(url, path)


def load_image(path):
    """Load an RGB uint8 image as a NumPy array (H, W, 3)."""
    return np.array(Image.open(path).convert("RGB"))


def _class_color(cls_id):
    rng = np.random.RandomState(int(cls_id) * 17 + 3)
    return rng.randint(40, 230, size=3) / 255.0


def visualize_detections(
    image,
    boxes,
    scores=None,
    labels=None,
    names=None,
    title=None,
    max_dets=60,
    prompt=None,
):
    """Draw xyxy boxes. `names` maps class id → string."""
    fig, ax = plt.subplots(1, 1, figsize=(10, 7))
    ax.imshow(image)
    ax.axis("off")
    header = title or ""
    if prompt:
        header = (header + "  |  prompt: " + str(prompt)).strip(" |")
    if header:
        ax.set_title(header, fontsize=12)

    boxes = [] if boxes is None else list(boxes)[:max_dets]
    scores = [None] * len(boxes) if scores is None else list(scores)[:max_dets]
    labels = [None] * len(boxes) if labels is None else list(labels)[:max_dets]

    for box, score, label in zip(boxes, scores, labels):
        x1, y1, x2, y2 = [float(v) for v in box]
        cls_id = 0 if label is None else int(label)
        color = _class_color(cls_id)
        ax.add_patch(
            patches.Rectangle(
                (x1, y1),
                max(x2 - x1, 1.0),
                max(y2 - y1, 1.0),
                linewidth=2,
                edgecolor=color,
                facecolor="none",
            )
        )
        name = names.get(cls_id, str(cls_id)) if isinstance(names, dict) else (str(label) if label is not None else "")
        caption = name if score is None else f"{name} {float(score):.2f}"
        ax.text(
            x1,
            max(y1 - 4, 12),
            caption,
            color="white",
            fontsize=9,
            bbox=dict(facecolor=color, edgecolor="none", pad=2, alpha=0.85),
        )
    fig.tight_layout()
    plt.show()
    return fig


def annotate_frame(image, boxes, scores=None, labels=None, names=None, max_dets=60):
    """Draw xyxy boxes on an RGB frame; returns RGB uint8 array (no matplotlib)."""
    out = np.asarray(image).copy()
    boxes = [] if boxes is None else list(boxes)[:max_dets]
    scores = [None] * len(boxes) if scores is None else list(scores)[:max_dets]
    labels = [None] * len(boxes) if labels is None else list(labels)[:max_dets]

    for box, score, label in zip(boxes, scores, labels):
        x1, y1, x2, y2 = [int(round(float(v))) for v in box]
        cls_id = 0 if label is None else int(label)
        rgb = (_class_color(cls_id) * 255).astype(np.uint8)
        color = (int(rgb[2]), int(rgb[1]), int(rgb[0]))  # OpenCV uses BGR
        cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)
        name = names.get(cls_id, str(cls_id)) if isinstance(names, dict) else (str(label) if label is not None else "")
        caption = name if score is None else f"{name} {float(score):.2f}"
        (tw, th), _ = cv2.getTextSize(caption, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
        cv2.rectangle(out, (x1, max(y1 - th - 6, 0)), (x1 + tw + 4, y1), color, -1)
        cv2.putText(out, caption, (x1 + 2, max(y1 - 4, th)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1, cv2.LINE_AA)
    return out


## 4. Load an example image


In [ ]:
# ==========================================
# INSTRUCTOR PROVIDED — DO NOT MODIFY
# ==========================================

IMAGE_PATH = download_image(SAMPLE_IMAGES["bus"], "bus.jpg")
image = load_image(IMAGE_PATH)
print("image:", image.shape, image.dtype)
plt.figure(figsize=(8, 6))
plt.imshow(image)
plt.axis("off")
plt.title("Input image")
plt.show()


## 5–6. The YOLO pipeline

```text
Image
  → preprocess (letterbox + normalize)
  → Backbone          # visual features at several resolutions
  → Neck / FPN+PAN    # fuse those scales
  → Detection head    # dense box + class scores
  → Decode            # turn raw tensors into boxes
  → NMS               # remove duplicate boxes
  → Final detections
```

Why each step exists:

| Stage | Why it exists |
|---|---|
| Preprocess | The network expects a fixed square tensor. Letterboxing preserves aspect ratio. |
| Backbone | Turns pixels into features (edges → parts → objects). |
| Neck | Small objects live on high-resolution maps; large objects on low-resolution maps. Fusion helps both. |
| Head | At every location, predict: *what* is here, *where* is the box, *how confident*? |
| Decode | Convert the network’s internal box format into pixel coordinates. |
| NMS | Dense prediction produces many boxes for the same object. Keep the best one. |


In [ ]:
# ==========================================
# INSTRUCTOR PROVIDED — DO NOT MODIFY
# ==========================================

from dataclasses import dataclass

from ultralytics import YOLO
from ultralytics.utils.ops import non_max_suppression, xywh2xyxy

YOLO_WEIGHTS = "yolov8n.pt"  # nano: fast enough for a live workshop
IMGSZ = 640

print("Loading", YOLO_WEIGHTS, "...")
yolo = YOLO(YOLO_WEIGHTS)
yolo.to(DEVICE)
yolo_nn = yolo.model.eval()
CLASS_NAMES = dict(yolo.names)
print("Classes:", len(CLASS_NAMES), "| device:", DEVICE)


def _module_indices(nn):
    backbone_end, detect_i = None, None
    for i, m in enumerate(nn.model):
        if m.__class__.__name__ == "SPPF":
            backbone_end = i
        if m.__class__.__name__ in {"Detect", "v10Detect"}:
            detect_i = i
    if detect_i is None:
        detect_i = len(nn.model) - 1
    if backbone_end is None:
        backbone_end = max(detect_i - 13, 0)
    return backbone_end, detect_i


BACKBONE_END, DETECT_IDX = _module_indices(yolo_nn)
print(f"YOLO split: backbone 0–{BACKBONE_END}, neck {BACKBONE_END + 1}–{DETECT_IDX - 1}, head {DETECT_IDX}")


def _forward_layers(nn, x, start, end, y):
    """Run YOLO layers [start, end] with the official skip-connection rule."""
    for i in range(start, end + 1):
        m = nn.model[i]
        if m.f != -1:
            x = y[m.f] if isinstance(m.f, int) else [x if j == -1 else y[j] for j in m.f]
        x = m(x)
        y.append(x if m.i in nn.save else None)
    return x, y


def letterbox(image, new_shape=IMGSZ):
    h, w = image.shape[:2]
    r = min(new_shape / h, new_shape / w)
    new_unpad = (int(round(w * r)), int(round(h * r)))
    dw, dh = (new_shape - new_unpad[0]) / 2, (new_shape - new_unpad[1]) / 2
    resized = cv2.resize(image, new_unpad, interpolation=cv2.INTER_LINEAR) if (w, h) != new_unpad else image
    top, bottom = int(round(dh - 0.1)), int(round(dh + 0.1))
    left, right = int(round(dw - 0.1)), int(round(dw + 0.1))
    padded = cv2.copyMakeBorder(resized, top, bottom, left, right, cv2.BORDER_CONSTANT, value=(114, 114, 114))
    return padded, {"orig_hw": (h, w), "pad": (top, left), "ratio": r, "imgsz": padded.shape[:2]}


def scale_boxes_to_original(boxes, meta):
    top, left = meta["pad"]
    r = meta["ratio"]
    h, w = meta["orig_hw"]
    boxes = boxes.clone() if torch.is_tensor(boxes) else torch.tensor(boxes, dtype=torch.float32)
    boxes[:, [0, 2]] -= left
    boxes[:, [1, 3]] -= top
    boxes /= r
    boxes[:, [0, 2]] = boxes[:, [0, 2]].clamp(0, w)
    boxes[:, [1, 3]] = boxes[:, [1, 3]].clamp(0, h)
    return boxes


@dataclass
class YOLOFeatures:
    x: torch.Tensor
    cache: list


def preprocess_image(image):
    """Letterbox to 640 and convert RGB HWC uint8 → BCHW float tensor in [0, 1]."""
    padded, meta = letterbox(image)
    tensor = torch.from_numpy(padded).to(DEVICE)
    tensor = tensor.permute(2, 0, 1).float().unsqueeze(0) / 255.0
    return tensor, meta


@torch.no_grad()
def extract_features(tensor):
    """Backbone: multi-resolution CNN features, ending at SPPF."""
    x, y = _forward_layers(yolo_nn, tensor, 0, BACKBONE_END, [])
    return YOLOFeatures(x=x, cache=y)


@torch.no_grad()
def build_multiscale_features(backbone_out):
    """Neck / FPN+PAN: fuse backbone maps so small and large objects share context."""
    x, y = _forward_layers(yolo_nn, backbone_out.x, BACKBONE_END + 1, DETECT_IDX - 1, backbone_out.cache)
    detect = yolo_nn.model[DETECT_IDX]
    feats = [y[j] for j in detect.f]
    return feats


@torch.no_grad()
def predict_boxes_and_classes(multiscale_feats):
    """Detection head: dense box + class scores at every spatial location / scale."""
    detect = yolo_nn.model[DETECT_IDX]
    out = detect(multiscale_feats)
    pred = out[0] if isinstance(out, (list, tuple)) else out
    return pred


def decode_predictions(raw_pred, meta, conf_thres=0.01):
    """Convert raw head output to xyxy boxes. No NMS yet — overlapping boxes remain."""
    pred = raw_pred[0] if raw_pred.dim() == 3 else raw_pred
    pred = pred.transpose(0, 1)  # (N, 4+nc)
    xywh, cls_scores = pred[:, :4], pred[:, 4:]
    conf, cls = cls_scores.max(1)
    keep = conf > conf_thres
    boxes = xywh2xyxy(xywh[keep])
    boxes = scale_boxes_to_original(boxes, meta)
    return {
        "boxes": boxes.cpu(),
        "scores": conf[keep].cpu(),
        "labels": cls[keep].cpu(),
        "names": CLASS_NAMES,
    }


def apply_nms(decoded, conf_thres=0.25, iou_thres=0.45):
    """Keep the strongest box among overlapping predictions of the same object."""
    boxes = decoded["boxes"]
    if len(boxes) == 0:
        return decoded
    # Rebuild the (1, 6) tensor NMS expects: xyxy, score, class — already decoded.
    dets = torch.cat(
        [boxes, decoded["scores"][:, None], decoded["labels"][:, None].float()],
        dim=1,
    )
    keep = []
    for cls_id in dets[:, 5].unique():
        m = dets[:, 5] == cls_id
        cls_dets = dets[m]
        order = cls_dets[:, 4].argsort(descending=True)
        cls_dets = cls_dets[order]
        while len(cls_dets):
            keep.append(cls_dets[0])
            if len(cls_dets) == 1:
                break
            iou = _box_iou(cls_dets[0, :4], cls_dets[1:, :4])
            cls_dets = cls_dets[1:][iou <= iou_thres]
    if not keep:
        empty = dets[:0]
        return {"boxes": empty[:, :4], "scores": empty[:, 4], "labels": empty[:, 5].long(), "names": CLASS_NAMES}
    keep = torch.stack(keep, 0)
    keep = keep[keep[:, 4] >= conf_thres]
    return {
        "boxes": keep[:, :4],
        "scores": keep[:, 4],
        "labels": keep[:, 5].long(),
        "names": CLASS_NAMES,
    }


def _box_iou(box, boxes):
    x1 = torch.maximum(box[0], boxes[:, 0])
    y1 = torch.maximum(box[1], boxes[:, 1])
    x2 = torch.minimum(box[2], boxes[:, 2])
    y2 = torch.minimum(box[3], boxes[:, 3])
    inter = (x2 - x1).clamp(0) * (y2 - y1).clamp(0)
    area_a = (box[2] - box[0]) * (box[3] - box[1])
    area_b = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    return inter / (area_a + area_b - inter + 1e-6)


def filter_class(dets, class_name):
    ids = [i for i, n in dets["names"].items() if n == class_name]
    if not ids:
        print("Unknown class:", class_name)
        return dets
    m = torch.isin(dets["labels"], torch.tensor(ids))
    return {**dets, "boxes": dets["boxes"][m], "scores": dets["scores"][m], "labels": dets["labels"][m]}


def show_dets(image, dets, title=None):
    visualize_detections(
        image,
        dets["boxes"].numpy() if torch.is_tensor(dets["boxes"]) else dets["boxes"],
        dets["scores"].numpy() if torch.is_tensor(dets["scores"]) else dets["scores"],
        dets["labels"].numpy() if torch.is_tensor(dets["labels"]) else dets["labels"],
        names=dets["names"],
        title=title,
    )


@torch.no_grad()
def run_full_pipeline(image, conf_thres=0.25, iou_thres=0.45):
    """Black-box demo. For the assembly exercise, call the named stages yourself."""
    tensor, meta = preprocess_image(image)
    features = extract_features(tensor)
    multiscale = build_multiscale_features(features)
    raw_pred = predict_boxes_and_classes(multiscale)
    decoded = decode_predictions(raw_pred, meta, conf_thres=0.01)
    detections = apply_nms(decoded, conf_thres=conf_thres, iou_thres=iou_thres)
    return {
        "tensor": tensor,
        "meta": meta,
        "features": features,
        "multiscale": multiscale,
        "raw_pred": raw_pred,
        "decoded": decoded,
        "detections": detections,
    }


def iter_video_frames(path, max_frames=None, stride=1):
    """Yield (frame_index, RGB uint8 array) from an MP4 or webcam file."""
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open video: {path}")
    idx = yielded = 0
    try:
        while True:
            ok, frame_bgr = cap.read()
            if not ok:
                break
            if idx % stride == 0:
                yield idx, cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
                yielded += 1
                if max_frames is not None and yielded >= max_frames:
                    break
            idx += 1
    finally:
        cap.release()


def run_pipeline_on_frame(frame, conf_thres=0.25, iou_thres=0.45):
    """Run the same YOLO stages as run_full_pipeline on one video frame."""
    result = run_full_pipeline(frame, conf_thres=conf_thres, iou_thres=iou_thres)
    dets = result["detections"]
    annotated = annotate_frame(
        frame,
        dets["boxes"].numpy() if torch.is_tensor(dets["boxes"]) else dets["boxes"],
        dets["scores"].numpy() if torch.is_tensor(dets["scores"]) else dets["scores"],
        dets["labels"].numpy() if torch.is_tensor(dets["labels"]) else dets["labels"],
        names=dets["names"],
    )
    return annotated, result


def process_video(
    video_path,
    out_path="yolo_video_out.mp4",
    conf_thres=0.25,
    iou_thres=0.45,
    max_frames=90,
    stride=1,
):
    """Run the YOLO pipeline on each frame and write an annotated MP4."""
    video_path = Path(video_path)
    probe = cv2.VideoCapture(str(video_path))
    fps = probe.get(cv2.CAP_PROP_FPS) or 25.0
    width = int(probe.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(probe.get(cv2.CAP_PROP_FRAME_HEIGHT))
    probe.release()

    out_path = Path(out_path)
    writer = cv2.VideoWriter(
        str(out_path),
        cv2.VideoWriter_fourcc(*"mp4v"),
        max(fps / stride, 1.0),
        (width, height),
    )
    counts = []
    for _, frame in iter_video_frames(video_path, max_frames=max_frames, stride=stride):
        annotated, result = run_pipeline_on_frame(frame, conf_thres=conf_thres, iou_thres=iou_thres)
        writer.write(cv2.cvtColor(annotated, cv2.COLOR_RGB2BGR))
        counts.append(len(result["detections"]["boxes"]))
    writer.release()
    return {"out_path": out_path, "frame_counts": counts, "fps": max(fps / stride, 1.0)}


print("YOLO helpers ready.")


## Instructor demo

This cell only calls a provided **one-liner** so Run all still shows a picture. The assembly exercise is to wire the named stages yourself — do not copy `run_full_pipeline`.


In [ ]:
# ==========================================
# INSTRUCTOR PROVIDED — DO NOT MODIFY
# ==========================================

DEMO = run_full_pipeline(image)
print("detections:", len(DEMO["detections"]["boxes"]))
show_dets(image, DEMO["detections"], title="YOLO demo (after NMS)")


## 7. Student assembly

Put the stages in order. Each function is already implemented — you are choosing **when** it runs, not **how**.


In [ ]:
# ==========================================
# STUDENT TASK
# ==========================================
# Set True after you replace every ??? with the correct function call.
RUN_STUDENT_ASSEMBLY = False

if RUN_STUDENT_ASSEMBLY:
    image = load_image(IMAGE_PATH)
    tensor, meta = preprocess_image(image)

    # TODO: assemble the one-stage pipeline in the correct order.
    # Available functions:
    #   extract_features, build_multiscale_features, predict_boxes_and_classes,
    #   decode_predictions, apply_nms, show_dets
    features = ???
    multiscale = ???
    raw_pred = ???
    decoded = ???
    detections = ???
    show_dets(image, detections, title="Student assembly")
    print("detections:", len(detections["boxes"]))
else:
    print('Skipping student assembly. The instructor demo above already ran the pipeline.')
    print('During the exercise: fill in the TODOs, then set RUN_STUDENT_ASSEMBLY = True.')


## 8–9. Experiments

Change **one** setting at a time and compare to the demo above.


In [ ]:
# ==========================================
# STUDENT TASK
# ==========================================

# Try: 0.05 (more boxes, more false positives) vs 0.6 (fewer, stricter)
CONF_THRES = 0.25
# Try: 0.1 (aggressive merge) vs 0.9 (keep overlapping boxes)
NMS_IOU = 0.45
# Set to a COCO name such as "person" or "bus", or None to keep all classes
KEEP_CLASS = None  # e.g. "person"

decoded_exp = decode_predictions(DEMO["raw_pred"], DEMO["meta"], conf_thres=0.01)
before_nms = {**decoded_exp}
before_nms_hi = apply_nms(decoded_exp, conf_thres=CONF_THRES, iou_thres=1.0)  # no real suppression
after_nms = apply_nms(decoded_exp, conf_thres=CONF_THRES, iou_thres=NMS_IOU)
if KEEP_CLASS:
    after_nms = filter_class(after_nms, KEEP_CLASS)

print(f"before NMS (conf>{CONF_THRES}, no IoU suppression):", len(before_nms_hi["boxes"]))
print(f"after  NMS (IoU={NMS_IOU}):", len(after_nms["boxes"]))
show_dets(image, before_nms_hi, title=f"Before NMS | conf={CONF_THRES}")
show_dets(image, after_nms, title=f"After NMS | conf={CONF_THRES}, IoU={NMS_IOU}")


Optional (if time): compare model size. `yolov8n.pt` is the default. `yolov8s.pt` is slower and usually more accurate. Reload with `YOLO("yolov8s.pt")` only if the GPU is free.


## 10. Optional: video input

Video is not a different model — it is the **same pipeline on every frame**. That is why real-time detectors like YOLO matter: they must stay fast when repeated 25–30 times per second.

We ship a short example clip (~260 KB) from the [Ultralytics assets](https://github.com/ultralytics/assets) repo. In Colab it downloads automatically; locally you can also use `examples/videos/solutions_ci_demo.mp4` if you cloned this workshop repo.

**Try:** raise `MAX_FRAMES` on a GPU, or set `STRIDE=2` to skip every other frame and finish faster.


In [ ]:
# ==========================================
# INSTRUCTOR PROVIDED — DO NOT MODIFY
# ==========================================

from IPython.display import Video, display

VIDEO_PATH = download_video(SAMPLE_VIDEOS["demo"], "workshop_demo.mp4")
print("video:", VIDEO_PATH)

frame_idx, first_frame = next(iter(iter_video_frames(VIDEO_PATH, max_frames=1)))
annotated_preview, frame_result = run_pipeline_on_frame(first_frame)
print(f"frame {frame_idx}: {len(frame_result['detections']['boxes'])} detections")
show_dets(first_frame, frame_result["detections"], title=f"Video frame {frame_idx}")

MAX_FRAMES = 60   # lower on CPU; raise on GPU
STRIDE = 2        # 1 = every frame, 2 = every other frame
VIDEO_OUT = process_video(
    VIDEO_PATH,
    out_path="yolo_video_out.mp4",
    conf_thres=CONF_THRES if "CONF_THRES" in globals() else 0.25,
    iou_thres=NMS_IOU if "NMS_IOU" in globals() else 0.45,
    max_frames=MAX_FRAMES,
    stride=STRIDE,
)
print("wrote:", VIDEO_OUT["out_path"], "| processed frames:", len(VIDEO_OUT["frame_counts"]))
print("detections per frame (first 5):", VIDEO_OUT["frame_counts"][:5])
display(Video(str(VIDEO_OUT["out_path"]), embed=True, width=640))


## 11. Think about it

1. Why can YOLO be **fast** even though it predicts thousands of boxes?
2. What goes wrong if you **remove NMS**?
3. A very low confidence threshold increases recall. What else does it increase?

Write short answers in a new markdown cell, or discuss with a neighbor. These are conceptual — no equations required.


## 12. Optional challenge

Visualize one neck feature map (for example `multiscale[0][0, 0].cpu()`) next to the detection image. You should see that high-resolution maps are sharper, while deeper maps are more abstract.


## 13. Summary

- YOLO is a **one-stage** detector: image → dense predictions → NMS.
- Backbone extracts features, neck fuses scales, head predicts boxes and classes.
- NMS exists because many nearby predictions describe the **same** object.

Next: **Faster R-CNN** — instead of scoring every location, first propose regions, then classify them.
